In [ ]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [ ]:
import torch
from src.utils.env import set_seed

set_seed(42)

torch.set_float32_matmul_precision("high")

In [ ]:
import pandas as pd

results_path = "../results.json"

results = pd.read_json(results_path, orient="records")

results.sample(10)

In [ ]:
def parse_path(path: str) -> dict:
    parts = path.split("/")
    model_name = parts[0]
    train_dataset = parts[1]
    layer_name = parts[2]
    exp_name = parts[3]

    return {"model_name": model_name, "train_dataset": train_dataset, "layer_name": layer_name, "exp_name": exp_name}


# now use parse_path to add corresponding columns to the dataframe
parsed = results["path"].apply(parse_path)
results = pd.concat([results, parsed.apply(pd.Series)], axis=1)

# results = results.drop(columns=["path", "file"])

results.sample(20)

In [ ]:
def parse_experiment(exp_name: str) -> dict:
    kl_value = None
    lr_value = None
    early_stop = True
    
    if "no-early-stop" in exp_name:
        early_stop = False
    
    if "baseline" in exp_name or "no-early-stop" in exp_name:
        lr_value = 0.1
        kl_value = 1.0
    
    if "small-lr" in exp_name:
        lr_value = 0.02
        kl_value = 1.0
    
    if "medium-lr" in exp_name:
        lr_value = 0.04
        kl_value = 1.0

    if "large-lr" in exp_name:
        lr_value = 0.25
        kl_value = 1.0

    if "small-kl" in exp_name:
        lr_value = 0.1
        kl_value = 0.5

    if "high-kl" in exp_name:
        lr_value = 0.1
        kl_value = 2.0

    # otherwise, the kl should appear as .._kl=<number>-..

    if "kl=" in exp_name:
        lr_value = 0.1

        kl_part = [part for part in exp_name.split("_") if part.startswith("kl=")]
        if not kl_part:
            raise ValueError(f"Could not find kl value in experiment name: {exp_name}")

        kl_value_str = kl_part[0].split("=")[1].split("-")[0] 
        kl_value = float(kl_value_str)

        kl_value = float(kl_value_str)
        
        
        
    if lr_value is None or kl_value is None:
        raise ValueError(f"Could not parse experiment name: {exp_name}")
        
    return {
        "kl_value": kl_value,
        "lr_value": lr_value,
        "early_stop": early_stop
    }


parsed = results["exp_name"].apply(parse_experiment)
results = pd.concat([results, parsed.apply(pd.Series)], axis=1)

results.sample(10)

In [ ]:
import torch

# res = torch.randn(size=(4096,)) * 0.01
# res = torch.rand(size=(4096,))
res = torch.empty(size=(4096,))

# res = torch.nn.init.uniform_(res, a=-1, b=1)
# res = torch.sign(torch.nn.init.uniform_(res, a=-1, b=1))  # sign init
# res = torch.nn.init.ones_(res)  # sign init
res = torch.nn.init.normal_(res)
# res = torch.nn.init.uniform(res) * torch.nn.init.normal(res)

print(torch.var(res / torch.norm(res)))

In [ ]:
# now please do the following: 
# group the data by (train_dataset, model_name, layer_name)

# i want to compare results across the following axes:
# how does the KL value affect the metrics (make sure that you only compare experiments with the same lr_value and early_stop value when comparing kl_value, and vice versa)
# how does lr_value affect the metrics (make sure that you only compare experiments with the same kl_value and early_stop value when comparing lr_value, and vice versa)
# how does early stopping affect the metrics (make sure that you only compare experiments with the same kl_value and lr_value when comparing early stopping, and vice versa)

In [ ]:
results.columns

In [ ]:
results["train_dataset"].unique()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

x_axis = "kl_value"
metric_names = ["top1_acc", "top10_agr", "proj_l2_rel", "kl_div"]  # change this to the desired metric names

In [ ]:
# check effect of early stopping
filtered = results[(results["lr_value"] == 0.1) & (results["kl_value"] == 1.0)]
x_axis = "early_stop"

# Group by train_dataset, model_name
grouped = filtered.groupby(['train_dataset', 'model_name'])

# Plot effect of x_axis parameter per layer for all metrics
for name, group in grouped:
    sub_group = group[(group['metric'].isin(metric_names))]
    if not sub_group.empty and len(sub_group[x_axis].unique()) > 1:
        col_wrap = min(4, len(metric_names))
        g = sns.FacetGrid(sub_group, col='metric', hue='layer_name', col_wrap=col_wrap, sharey=False, height=6)
        g.map_dataframe(sns.scatterplot, x=x_axis, y='value', alpha=0.75, marker='o')
        g.map_dataframe(sns.lineplot, x=x_axis, y='value', estimator='mean', alpha=1, linewidth=2, err_kws={'alpha': 0.2})
        g.add_legend()
        g.fig.suptitle(f'{x_axis} effect for train-dataset={name[0]}, model={name[1]}', y=1.02)
        plt.show()

In [ ]:
# check effect of early stopping
filtered = results[(results["kl_value"] == 1.0) & (results["early_stop"] == True)]
x_axis = "lr_value"

# Group by train_dataset, model_name
grouped = filtered.groupby(['train_dataset', 'model_name'])

# Plot effect of x_axis parameter per layer for all metrics
for name, group in grouped:
    sub_group = group[(group['metric'].isin(metric_names))]
    if not sub_group.empty and len(sub_group[x_axis].unique()) > 1:
        col_wrap = min(4, len(metric_names))
        g = sns.FacetGrid(sub_group, col='metric', hue='layer_name', col_wrap=col_wrap, sharey=False, height=6)
        g.map_dataframe(sns.scatterplot, x=x_axis, y='value', alpha=0.75, marker='o')
        g.map_dataframe(sns.lineplot, x=x_axis, y='value', estimator='mean', alpha=1, linewidth=2, err_kws={'alpha': 0.2})
        g.add_legend()
        g.fig.suptitle(f'{x_axis} effect for train-dataset={name[0]}, model={name[1]}', y=1.02)
        plt.show()

In [ ]:
filtered = results[(results["lr_value"] == 0.1) & (results["early_stop"] == True)]

filtered = filtered[filtered["kl_value"] < 10.0]

# Group by train_dataset, model_name
grouped = filtered.groupby(['train_dataset', 'model_name'])

# Plot effect of x_axis parameter per layer for all metrics
for name, group in grouped:
    sub_group = group[(group['metric'].isin(metric_names))]
    if not sub_group.empty and len(sub_group[x_axis].unique()) > 1:
        col_wrap = min(4, len(metric_names))
        g = sns.FacetGrid(sub_group, col='metric', hue='layer_name', col_wrap=col_wrap, sharey=False, height=6)
        g.map_dataframe(sns.scatterplot, x=x_axis, y='value', alpha=0.75, marker='o')
        g.map_dataframe(sns.lineplot, x=x_axis, y='value', estimator='mean', alpha=1, errorbar=None, linewidth=2)
        g.add_legend()
        g.fig.suptitle(f'{x_axis} effect for train-dataset={name[0]}, model={name[1]}', y=1.02)
        plt.show()

In [ ]:
print(filtered["benchmark"].unique())

In [ ]:
metric_names = ["proj_l2_rel", "kl_div"]

filtered = filtered[filtered["benchmark"] == "eval-lmsys-1m"]

# Group by train_dataset, model_name
grouped = filtered.groupby(['model_name'])


# Plot effect of x_axis parameter per layer for all metrics
for name, group in grouped:
    sub_group = group[(group['metric'].isin(metric_names))]
    if not sub_group.empty and len(sub_group[x_axis].unique()) > 1:
        col_wrap = min(4, len(metric_names))
        g = sns.FacetGrid(sub_group, col='metric', hue='layer_name', col_wrap=col_wrap, sharey=False, height=6)
        g.map_dataframe(sns.scatterplot, x=x_axis, y='value', alpha=0.85, marker='o')
        g.map_dataframe(sns.lineplot, x=x_axis, y='value', estimator='mean', alpha=1, linewidth=2, err_kws={'alpha': 0.2})
        g.add_legend()
        g.fig.suptitle(f'{x_axis} effect for model={name[0]}', y=1.02)
        plt.show()

In [ ]:
# CONCLUSIONS:
# KL >= 10 is not good at all
# training dataset is important
# early stopping is also important, especially for Qwen model...
# set early stopping to 200...
# initialization of w is important. test uniform vs normal initialization. note that mean and std do not matter, since we normalize w to have norm 1.
# use lr=0.1

# TODO:
# - run training on slim-orca and hh-rlhf with kl=0.5
# - benchmark all training datasets with kl=0.5